# NextFlix — EDA & Modeling Notebook
**AI/ML Internship Task — Aakash Group**

This notebook covers the full exploratory analysis and modeling rationale.
For production training, use `python train.py`.

| Step | Description |
|------|-------------|
| 1 | Dataset exploration & EDA |
| 2 | Text preprocessing pipeline |
| 3 | TF-IDF vs BoW comparison |
| 4 | Recommendation demos |
| 5 | Evaluation metrics |

In [ ]:
import re, sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
sns.set_style('darkgrid')

print('Imports Successful')

## 1. Dataset Exploration

In [ ]:
# dataset  = load_dataset('jquigl/imdb-genres')
# train_df = pd.DataFrame(dataset['train'])
# val_df   = pd.DataFrame(dataset['validation'])

train_df = pd.read_csv('../dataset/train.csv')
val_df = pd.read_csv('../dataset/validation.csv')

print(f'Train : {train_df.shape}  |  Val : {val_df.shape}')
print(f'Columns: {list(train_df.columns)}')

In [ ]:
train_df.rename(columns={'movie title - year': 'title'}, inplace=True)
train_df.head(3)

In [ ]:
print('Missing values:')
print(train_df.isnull().sum())
print(f'\nDuplicate titles: {train_df["title"].duplicated().sum()}')

In [ ]:
# Description length distribution
train_df['desc_len'] = train_df['description'].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df['desc_len'].hist(bins=50, ax=axes[0], color='#f5c842', edgecolor='none')
axes[0].set_title('Description Word Count Distribution')
axes[0].set_xlabel('Words')

# Genre distribution
genre_col = next((c for c in ['expanded-genres','genre','genres'] if c in train_df.columns), None)
if genre_col:
    genre_counts = (
        train_df[genre_col].dropna()
        .str.split(r'[,|/]').explode().str.strip()
        .value_counts().head(12)
    )
    genre_counts.plot(kind='barh', ax=axes[1], color='#e8834a')
    axes[1].set_title('Top 12 Genres')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()
print(f'Avg description length: {train_df["desc_len"].mean():.1f} words')

## 2. Text Preprocessing
Pipeline: **Lowercase → Remove punctuation → Remove stopwords → Lemmatise**

In [ ]:
def clean_text(text):
    if not isinstance(text, str) or not text.strip(): return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(
        lemmatizer.lemmatize(w) for w in text.split()
        if w not in STOP_WORDS and len(w) > 2
    )

# Demo
sample = train_df['description'].iloc[0]
print('ORIGINAL:\n', sample)
print('\nCLEANED:\n', clean_text(sample))

train_df['clean_description'] = train_df['description'].apply(clean_text)
train_df = train_df[train_df['clean_description'].str.strip()!=''].reset_index(drop=True)
print(f'\nRetained: {len(train_df):,} movies')

## 3. TF-IDF vs BoW Comparison

| Method | Weights | Behaviour |
|--------|---------|----------|
| **BoW** | Raw count | 'film' and 'story' get high weight everywhere |
| **TF-IDF** | count × log(N/df) | Down-weights common words, up-weights rare discriminating terms |

In [ ]:
# Build both
bow_vec  = CountVectorizer(max_features=20_000, ngram_range=(1,2), min_df=2)
tfidf_vec = TfidfVectorizer(max_features=20_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)

bow_mat   = bow_vec.fit_transform(train_df['clean_description'])
tfidf_mat = tfidf_vec.fit_transform(train_df['clean_description'])

print(f'BoW   matrix : {bow_mat.shape}')
print(f'TF-IDF matrix: {tfidf_mat.shape}')

# Compare top terms for a sample movie
idx = 0
movie_title = train_df['title'].iloc[idx]

bow_scores   = dict(zip(bow_vec.get_feature_names_out(),   bow_mat[idx].toarray()[0]))
tfidf_scores = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_mat[idx].toarray()[0]))

top_bow   = sorted(bow_scores.items(),   key=lambda x:-x[1])[:10]
top_tfidf = sorted(tfidf_scores.items(), key=lambda x:-x[1])[:10]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
terms_b, scores_b = zip(*top_bow)
terms_t, scores_t = zip(*top_tfidf)
axes[0].barh(terms_b, scores_b, color='#6c63ff'); axes[0].set_title(f'BoW — "{movie_title}"'); axes[0].invert_yaxis()
axes[1].barh(terms_t, scores_t, color='#f5c842'); axes[1].set_title(f'TF-IDF — "{movie_title}"'); axes[1].invert_yaxis()
plt.tight_layout()
plt.show()
print('→ TF-IDF surfaces more specific, meaningful terms')

## 4. Recommendation Demos

In [ ]:
title_index = pd.Series(train_df.index, index=train_df['title'].str.lower())

def recommend_by_title(title, n=5):
    key = title.lower()
    if key not in title_index:
        return pd.DataFrame()

    idx = title_index[key]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    scores = cosine_similarity(tfidf_mat[idx], tfidf_mat).flatten()
    scores[idx] = -1

    top = scores.argsort()[::-1]
    top = [i for i in top if i < len(train_df)][:n]

    res = train_df.iloc[top][['title','description']].copy()
    res['score'] = scores[top].round(4)

    return res.reset_index(drop=True)

def recommend_by_query(query, n=5):
    vec = tfidf_vec.transform([clean_text(query)])
    scores = cosine_similarity(vec, tfidf_mat).flatten()
    
    top = scores.argsort()[::-1]
    top = [i for i in top if i < len(train_df)][:n]  # safe filter

    res = train_df.iloc[top][['title','description']].copy()
    res['score'] = scores[top].round(4)
    
    return res.reset_index(drop=True)

# Demo
demo = train_df['title'].iloc[0]
print(f'Similar to "{demo}":'); display(recommend_by_title(demo)[['title','score']])
print()
print('Space adventure query:'); display(recommend_by_query('space adventure unlikely hero saving galaxy')[['title','score']])

In [ ]:
# Ensure train_df index matches TF-IDF
train_df = train_df.reset_index(drop=True)
tfidf_mat = tfidf_vec.fit_transform(train_df['description'])

# Build title lookup (lowercase)
title_index = pd.Series(train_df.index, index=train_df['title'].str.lower())

def recommend_by_title(title, n=5):
    """
    Recommend movies similar to a given title.
    Returns top n results with title, description, and similarity score.
    """
    key = title.lower()
    if key not in title_index:
        return pd.DataFrame(columns=['title','description','score'])
    
    idx = title_index[key]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]  # take first match if duplicate titles exist
    
    # Compute similarity
    scores = cosine_similarity(tfidf_mat[idx], tfidf_mat).flatten()
    scores[idx] = -1  # ignore self

    # Get top n valid indices
    top = scores.argsort()[::-1]
    top = [i for i in top if i < len(train_df)][:n]

    res = train_df.iloc[top][['title','description']].copy()
    res['score'] = scores[top].round(4)
    return res.reset_index(drop=True)

def recommend_by_query(query, n=5):
    """
    Recommend movies based on a free-text query.
    Returns top n results with title, description, and similarity score.
    """
    vec = tfidf_vec.transform([clean_text(query)])
    scores = cosine_similarity(vec, tfidf_mat).flatten()

    top = scores.argsort()[::-1]
    top = [i for i in top if i < len(train_df)][:n]

    res = train_df.iloc[top][['title','description']].copy()
    res['score'] = scores[top].round(4)
    return res.reset_index(drop=True)

# Demo
demo = train_df['title'].iloc[0]
print(f'Similar to "{demo}":'); display(recommend_by_title(demo)[['title','score']])
print()
print('Space adventure query:'); display(recommend_by_query('space adventure unlikely hero saving galaxy')[['title','score']])

## 5. Evaluation

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **ILS** | avg pairwise cosine sim in top-N list | Lower = more diverse |
| **Avg score** | mean cosine sim to query | Higher = more relevant |
| **Coverage** | % of catalogue ever recommended | Higher = less popularity bias |

In [ ]:
sample_titles = train_df['title'].sample(50, random_state=42).tolist()
ils_scores, avg_scores, seen = [], [], set()

for title in sample_titles:
    recs = recommend_by_title(title)
    if recs.empty: 
        continue

    seen.update(recs['title'].tolist())
    avg_scores.append(recs['score'].mean())

    # Build safe flat list of indices
    indices = []
    for t in recs['title']:
        idx = title_index.get(t.lower())
        if idx is None:
            continue
        if isinstance(idx, pd.Series):
            idx = idx.iloc[0]
        indices.append(idx)

    if len(indices) >= 2:
        sub = tfidf_mat[indices]           # now safe
        sim = cosine_similarity(sub)       # intra-list similarity
        np.fill_diagonal(sim, 0)
        ils_scores.append(sim.sum() / (len(indices)*(len(indices)-1)))

print(f'Mean ILS (diversity, lower=better) : {np.mean(ils_scores):.4f}')
print(f'Mean similarity score              : {np.mean(avg_scores):.4f}')
print(f'Catalogue coverage                 : {len(seen)/len(train_df)*100:.1f}%')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(ils_scores, bins=15, color='#6c63ff', edgecolor='none')
axes[0].axvline(np.mean(ils_scores), color='red', linestyle='--', label=f'Mean={np.mean(ils_scores):.3f}')
axes[0].set_title('Intra-List Similarity Distribution'); axes[0].legend()
axes[1].hist(avg_scores, bins=15, color='#f5c842', edgecolor='none')
axes[1].axvline(np.mean(avg_scores), color='red', linestyle='--', label=f'Mean={np.mean(avg_scores):.3f}')
axes[1].set_title('Avg Recommendation Score'); axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Limitations & Improvements

### Limitations
1. **No semantic understanding** — 'car chase' ≠ 'automobile pursuit' in TF-IDF space
2. **Cold-start** — very short descriptions produce sparse, poor-quality vectors
3. **No personalisation** — purely content-based; ignores user history
4. **Scalability** — full cosine similarity is O(n²); impractical at > 100k movies
5. **Description quality bias** — verbose summaries dominate recommendations

### Improvements
| Approach | Benefit |
|----------|---------|
| Sentence Transformers (SBERT) | Semantic similarity, handles paraphrases |
| BM25 instead of TF-IDF | Better short-query ranking |
| FAISS / Annoy ANN search | O(log n) retrieval at scale |
| Collaborative filtering (ALS) | Adds user-preference signal |
| Hybrid model (content + CF) | Best-of-both accuracy |